# Handling the Response

Hello and welcome back. In our last lectures, we made our first successful API call. That call returned a `response` object. Now, we need to understand what's inside that object and how to extract the information we need.

An API is only useful if you can understand what it sends back. The OpenAI API provides a rich, structured response that contains not only the AI's generated text but also valuable metadata about the request.

Let's make an API call and then dissect the response piece by piece.

## Making a Call to Get a Response Object

First, let's set up our environment and make a simple API call, just as we did before. This time, we will capture the entire response object in a variable so we can inspect it.

In [3]:
import litellm
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = "openai/gpt-4o-mini"
MESSAGES = [
    {
        "role": "user",
        "content": "Write a short, 3-line poem about Python programming."
    }
]

In [4]:
response_object = litellm.completion(
    model=MODEL_NAME,
    messages=MESSAGES
)

## Dissecting the Response Object

The `response` object is not just a simple string. It's a structured object (a Pydantic model) that contains a wealth of useful information. Let's print out some of the top-level attributes.

In [7]:
print(f"ID: {response_object.id}")
print(f"Model used: {response_object.model}")
print(f"Created timestamp {response_object.created}")

# --- Usage information ----
usage_data = response_object.usage
print("\n--- Usage Data ----")
print(f"Prompt tokens (input): {usage_data.prompt_tokens}")
print(f"Completion tokens (output): {usage_data.completion_tokens}")
print(f"Total tokens: {usage_data.total_tokens}")

ID: chatcmpl-E4YzcV3vSA2VQDOBJBBi7XDhcBMMw
Model used: gpt-4o-mini-2024-07-18
Created timestamp 1784756532

--- Usage Data ----
Prompt tokens (input): 19
Completion tokens (output): 33
Total tokens: 52


## Getting the Content via `choices[0].message.content`

The most important part of the response is the AI's actual message. This is located inside a list called `choices`. With the default settings, this list will always have one item.

Let's follow the path to the content: **`response.choices[0].message.content`**

In [9]:
print(response_object.choices[0].message.content)

In lines of code, logic dances free,  
With indents that sing, and loops that agree,  
Python weaves dreams, as clear as the sea.


## Handling Multiple Choices (n > 1)

Now we can answer the question: why is `choices` a list? Because you can ask the model to generate multiple different completions for the same prompt by setting the parameter `n`. This is perfect for brainstorming, generating alternative phrasings, or creating variations for A/B testing.

Let's ask the model for **three** different poems for the same prompt.

> Cost Warning: Setting `n=3` will generate three separate completions. Your `completion_tokens` in the usage object will be roughly three times higher, and you will be billed accordingly.
>

In [10]:
multi_choice_response = litellm.completion(
    model=MODEL_NAME,
    messages=MESSAGES,
    n=3
)

In [18]:
for i, choice in enumerate(multi_choice_response.choices):
    print(f"\n--- Choice {i+1} ---")
    print(choice.message.content)

print("\n--- Cost component ---")
print(f"Input tokens for three choices: {multi_choice_response.usage.prompt_tokens}")
print(f"Output tokens for three choices: {multi_choice_response.usage.completion_tokens}")
print(f"Total tokens for three choices: {multi_choice_response.usage.total_tokens}")


--- Choice 1 ---
In code's embrace, logic takes flight,  
Indentations weave through day and night,  
Python whispers, "Create with delight."

--- Choice 2 ---
In lines of code, a river flows,  
With loops and lists, the logic grows,  
Python's grace, where knowledge glows.

--- Choice 3 ---
In Python’s syntax, clarity flows,  
Indentations guide where logic goes,  
Code whispers secrets that only it knows.

--- Cost component ---
Input tokens for three choices: 19
Output tokens for three choices: 81
Total tokens for three choices: 100
